# Test en local

## /health

In [1]:
import requests

BASE_URL = "http://127.0.0.1:8000"

r = requests.get(f"{BASE_URL}/health", timeout=10)
print("status_code:", r.status_code)
print("json:", r.json())

status_code: 200
json: {'status': 'ok'}


## /predict

In [ ]:
import requests

BASE_URL = "http://127.0.0.1:8000"

payload = {
    "features": {
        # Exemple fictif :
        "EXT_SOURCE_1": 0.5,
        "EXT_SOURCE_2": 0.7,
        "EXT_SOURCE_3": 0.2
    }
}

r = requests.post(f"{BASE_URL}/predict", json=payload, timeout=30)
print("status_code:", r.status_code)
print("json:", r.json())

status_code: 200
json: {'approved': False, 'probability': 0.09050889041337443, 'threshold': 0.09}


In [15]:
import joblib
from pathlib import Path

MODEL_PATH = Path("api/models/model.pkl")
model = joblib.load(MODEL_PATH)

cols = list(model.feature_names_in_)
print("nb colonnes attendues:", len(cols))
print(cols[:50])  # aperçu

nb colonnes attendues: 184
['SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'ORGANIZATION_TYPE', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMM

In [16]:
print(type(model))
print(model)

<class 'imblearn.pipeline.Pipeline'>
Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler(with_mean=False))]),
                                                  ['CNT_CHILDREN',
                                                   'AMT_INCOME_TOTAL',
                                                   'AMT_CREDIT', 'AMT_ANNUITY',
                                                   'AMT_GOODS_PRICE',
                                                   'REGION_POPULATION_RELATIVE',
                                                   'DAYS_BIRTH',
                                                   'DAYS_EMPLOYED',
             

### Ligne réelle (application_test.csv)

In [1]:
import pandas as pd
import numpy as np
import requests

BASE_URL = "http://127.0.0.1:8000"

df = pd.read_csv("data/raw/application_test.csv")
row = df.sample(1).iloc[0].drop(labels=["SK_ID_CURR"], errors="ignore")
# row = df.sample(1, random_state=42).iloc[0].drop(labels=["SK_ID_CURR"], errors="ignore")

print(row)

# Remplacer NaN -> None (JSON null)
features = row.where(pd.notnull(row), None).to_dict()

payload = {"features": features}

r = requests.post(f"{BASE_URL}/predict", json=payload, timeout=30)
print(r.status_code)
print(r.json())

NAME_CONTRACT_TYPE            Cash loans
CODE_GENDER                            F
FLAG_OWN_CAR                           N
FLAG_OWN_REALTY                        N
CNT_CHILDREN                           1
                                 ...    
AMT_REQ_CREDIT_BUREAU_DAY            0.0
AMT_REQ_CREDIT_BUREAU_WEEK           0.0
AMT_REQ_CREDIT_BUREAU_MON            0.0
AMT_REQ_CREDIT_BUREAU_QRT            0.0
AMT_REQ_CREDIT_BUREAU_YEAR           0.0
Name: 21014, Length: 120, dtype: object
200
{'approved': False, 'probability': 0.16184293677502717, 'threshold': 0.09}


# Api déployée (Render)

## /health

In [10]:
import requests

BASE_URL = "https://api-oc-projet7.onrender.com"

r = requests.get(f"{BASE_URL}/health", timeout=30)
print("status_code:", r.status_code)
print("json:", r.json())

status_code: 200
json: {'status': 'ok'}


## /predict

In [11]:
import pandas as pd
import requests

BASE_URL = "https://api-oc-projet7.onrender.com"

df = pd.read_csv("data/raw/application_test.csv")
row = df.sample(1, random_state=42).iloc[0].drop(labels=["SK_ID_CURR"], errors="ignore")

# JSON n'accepte pas NaN => on convertit en None
features = row.where(pd.notnull(row), None).to_dict()

payload = {"features": features}

r = requests.post(f"{BASE_URL}/predict", json=payload, timeout=60)
print("status_code:", r.status_code)
print("json/text:", r.json() if r.headers.get("content-type","").startswith("application/json") else r.text)

status_code: 200
json/text: {'approved': True, 'probability_default': 0.019930073644163836}
